In [ ]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [ ]:
!pip show ultralytics

Name: ultralytics
Version: 8.4.193
Summary: Ultralytics YOLO 🚀 for SOTA object detection, instance segmentation, semantic segmentation, depth estimation, classification, pose estimation, oriented object detection, and multi-object tracking.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: matplotlib, numpy, nvidia-ml-py, opencv-python, pillow, polars, psutil, pyyaml, requests, torch, torchvision, ultralytics-platform, ultralytics-thop
Required-by: 


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

from pathlib import Path
import pandas as pd

DATA_DIR = Path('/content/gdrive/MyDrive/Optativa1/data')

Mounted at /content/gdrive


# **Definir la ruta correcta**
# **Aquí está uno de los errores importantes del código original.**
David tenía esta ruta:

DATA_DIR = '/content/gdrive/My Drive/Optativa_1/modelo/data/train'

Esa ruta apunta únicamente a `train`. Para esta estructura de clasificación, necesitamos indicar la carpeta `data`, que contiene las tres divisiones: `train`, `val` y `test`.

#### **Reemplaza el bloque anterior por este:**



In [ ]:
from pathlib import Path

DATA_DIR = Path(
    '/content/gdrive/MyDrive/data'
)

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"No se encontró el dataset en: {DATA_DIR}"
    )

print("Dataset encontrado correctamente.")
print("Contenido:", sorted(p.name for p in DATA_DIR.iterdir()))

FileNotFoundError: No se encontró el dataset en: /content/gdrive/MyDrive/modelo/data

# **Verificar las diez clases y contar las imágenes**
Este paso evita entrenar durante varios minutos para descubrir después que falta una carpeta o que una clase está vacía.

In [ ]:
from pathlib import Path

EXTENSIONES = {
    '.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'
}

TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR = DATA_DIR / 'val'
TEST_DIR = DATA_DIR / 'test'

for carpeta in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not carpeta.is_dir():
        raise FileNotFoundError(
            f"Falta la carpeta: {carpeta}"
        )

clases_train = {
    p.name for p in TRAIN_DIR.iterdir() if p.is_dir()
}

if len(clases_train) < 2:
    raise ValueError("Se necesitan al menos dos clases.")

for nombre_split, carpeta in [
    ('train', TRAIN_DIR),
    ('val', VAL_DIR),
    ('test', TEST_DIR)
]:
    clases_split = {
        p.name for p in carpeta.iterdir() if p.is_dir()
    }

    print(f"\n--- {nombre_split.upper()} ---")

    for clase in sorted(clases_split):
        cantidad = sum(
            1 for archivo in (carpeta / clase).rglob('*')
            if archivo.is_file()
            and archivo.suffix.lower() in EXTENSIONES
        )
        print(f'{clase}: {cantidad} imágenes')

    if clases_split != clases_train:
        raise ValueError(
            f"Las clases de {nombre_split} "
            "no coinciden con las de train."
        )


--- TRAIN ---
Anartia: 350 imágenes
Ascia: 350 imágenes
Battus: 350 imágenes
Danaus: 350 imágenes
Dryas: 350 imágenes
Heliconius: 350 imágenes
Morpho: 350 imágenes
Papilio: 350 imágenes
Phoebis: 350 imágenes
Vanessa: 350 imágenes

--- VAL ---
Anartia: 100 imágenes
Ascia: 100 imágenes
Battus: 100 imágenes
Danaus: 100 imágenes
Dryas: 100 imágenes
Heliconius: 100 imágenes
Morpho: 100 imágenes
Papilio: 100 imágenes
Phoebis: 100 imágenes
Vanessa: 100 imágenes

--- TEST ---
Anartia: 50 imágenes
Ascia: 50 imágenes
Battus: 50 imágenes
Danaus: 50 imágenes
Dryas: 50 imágenes
Heliconius: 50 imágenes
Morpho: 50 imágenes
Papilio: 50 imágenes
Phoebis: 50 imágenes
Vanessa: 50 imágenes


# **Comprobar si Colab dispone de una GPU:**

In [ ]:
import torch
import ultralytics
gpu_disponible = torch.cuda.is_available()
print("Versión de Ultralytics:", ultralytics.__version__)
print("GPU disponible:", gpu_disponible)

Versión de Ultralytics: 8.4.191
GPU disponible: True


#### Si el resultado de GPU es `False`, entra en **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**, si esa opción está disponible, y vuelve a ejecutar las celdas necesarias.

### **Encontrar los dos modelos dentro del Drive compartido**
Primero, ejecuta esta celda. Busca los archivos en `data`, incluyendo sus subcarpetas. Así no tenemos que adivinar si están directamente dentro de `data` o en una carpeta de pesos.

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/gdrive/MyDrive/data')

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"No se encuentra el dataset compartido: {DATA_DIR}"
    )

def localizar_checkpoint(nombre):
    # Priorizar el archivo directamente dentro de data.
    directo = DATA_DIR / nombre

    if directo.is_file():
        return directo

    # Si está dentro de una subcarpeta, buscarlo allí.
    encontrados = sorted(
        p for p in DATA_DIR.rglob(nombre)
        if p.is_file()
    )

    if len(encontrados) == 1:
        return encontrados[0]

    if len(encontrados) == 0:
        raise FileNotFoundError(
            f"No se encontró {nombre} dentro de {DATA_DIR}"
        )

    raise RuntimeError(
        f"Hay varias copias de {nombre}. "
        "Debes elegir la ruta correcta:\n" +
        "\n".join(str(p) for p in encontrados)
    )

BEST_PATH = localizar_checkpoint('best.pt')
LAST_PATH = localizar_checkpoint('last.pt')

print("Dataset:", DATA_DIR)
print("Mejor modelo:", BEST_PATH)
print("Último modelo:", LAST_PATH)

Dataset: /content/gdrive/MyDrive/data
Mejor modelo: /content/gdrive/MyDrive/data/best.pt
Último modelo: /content/gdrive/MyDrive/data/last.pt


# **Comparar los dos archivos `results.csv`: el original y el del reentrenamiento**

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('/content/gdrive/MyDrive/data')

archivos = {
    'ENTRENAMIENTO ORIGINAL': DATA_DIR / 'results.csv',
    'REENTRENAMIENTO': (
        DATA_DIR / 'runs' /
        'mariposas_reentreno_27-10-2026_20-07-20' /
        'results.csv'
    )
}

for titulo, ruta in archivos.items():
    print(f'\n{"=" * 20} {titulo} {"=" * 20}')

    if ruta.is_file():
        df = pd.read_csv(ruta)
        print("Columnas:", df.columns.tolist())
        print(df.to_string(index=False))
    else:
        print("No se encontró:", ruta)


==================== ENTRENAMIENTO ORIGINAL ====================
Columnas: ['epoch', 'time', 'train/loss', 'metrics/accuracy_top1', 'metrics/accuracy_top5', 'val/loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']
 epoch     time  train/loss  metrics/accuracy_top1  metrics/accuracy_top5  val/loss   lr/pg0   lr/pg1   lr/pg2
     1  259.932     1.55110                  0.910                  0.994   0.31760 0.000237 0.000237 0.000237
     2  346.338     0.51189                  0.942                  0.997   0.18678 0.000428 0.000428 0.000428
     3  432.777     0.41171                  0.940                  0.996   0.18128 0.000572 0.000572 0.000572
     4  518.586     0.40310                  0.949                  0.999   0.13498 0.000502 0.000502 0.000502
     5  604.764     0.31229                  0.967                  0.998   0.10662 0.000431 0.000431 0.000431
     6  691.459     0.27348                  0.962                  0.998   0.09779 0.000361 0.000361 0.000361
     7  777.965     0.2

# **Ejecutar el reentrenamiento**
Crea una celda nueva en Colab y ejecuta este código. Utiliza el `best.pt` compartido, incorpora EarlyStopping y guarda los resultados nuevos en otra carpeta sin sobrescribir los modelos originales.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from ultralytics import YOLO
import torch
import shutil

# 1. Rutas compartidas
DATA_DIR = Path('/content/gdrive/MyDrive/data')
BEST_PATH = DATA_DIR / 'best.pt'
LAST_PATH = DATA_DIR / 'last.pt'

# 2. Verificar archivos y dataset
assert BEST_PATH.is_file(), f"No se encontró: {BEST_PATH}"
assert LAST_PATH.is_file(), f"No se encontró: {LAST_PATH}"

for split in ['train', 'val', 'test']:
    assert (DATA_DIR / split).is_dir(), f"Falta: {split}"

# 3. Cargar el mejor modelo anterior
model = YOLO(str(BEST_PATH))

# 4. Configurar el hardware
gpu_disponible = torch.cuda.is_available()
device = 0 if gpu_disponible else "cpu"
epochs = 50 if gpu_disponible else 3
batch = 16 if gpu_disponible else 8

print("Modelo inicial:", BEST_PATH)
print("Modelo de respaldo:", LAST_PATH)
print("GPU disponible:", gpu_disponible)
print("Dispositivo:", device)
print("Épocas máximas:", epochs)

# 5. Reentrenar con EarlyStopping
results = model.train(
    data=str(DATA_DIR),
    epochs=epochs,
    imgsz=256,
    batch=batch,
    device=device,
    optimizer="AdamW",
    lr0=0.0003,
    patience=8,
    project="/content/runs",
    name="mariposas_reentreno",
    exist_ok=False,
    save=True,
    plots=True
)

# 6. Obtener la carpeta de resultados de Colab
carpeta_salida = Path(model.trainer.save_dir)

# 7. Registrar la fecha y hora al finalizar el entrenamiento
fecha_fin = datetime.now(
    ZoneInfo("America/Bogota")
).strftime("%d-%m-%Y_%H-%M-%S")

# 8. Crear una carpeta nueva en el Drive compartido
DESTINO = (
    DATA_DIR / "runs" /
    f"mariposas_reentreno_{fecha_fin}"
)

DESTINO.parent.mkdir(parents=True, exist_ok=True)

if DESTINO.exists():
    raise FileExistsError(
        f"La carpeta ya existe: {DESTINO}"
    )

# 9. Copiar todos los resultados al Drive
shutil.copytree(carpeta_salida, DESTINO)

print("\nREENTRENAMIENTO FINALIZADO")
print("Carpeta compartida:", DESTINO)
print("Mejor modelo:", DESTINO / "weights" / "best.pt")
print("Último modelo:", DESTINO / "weights" / "last.pt")

Modelo inicial: /content/gdrive/MyDrive/data/best.pt
Modelo de respaldo: /content/gdrive/MyDrive/data/last.pt
GPU disponible: True
Dispositivo: 0
Épocas máximas: 50
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gdrive/MyDrive/data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=

# **COMPARAR AMBOS MODELOS (ORIGINAL vs NUEVO):**

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch

# Dataset compartido
DATA_DIR = Path('/content/gdrive/MyDrive/data')

# Modelo del primer entrenamiento
MODELO_ORIGINAL = DATA_DIR / 'best.pt'

# Mejor modelo del reentrenamiento
MODELO_NUEVO = (
    DATA_DIR
    / 'runs'
    / 'mariposas_reentreno_27-10-2026_20-07-20'
    / 'weights'
    / 'best.pt'
)

# Comprobar que existen los dos modelos
assert MODELO_ORIGINAL.is_file(), (
    f"No se encuentra el modelo original: {MODELO_ORIGINAL}"
)
assert MODELO_NUEVO.is_file(), (
    f"No se encuentra el nuevo modelo: {MODELO_NUEVO}"
)

device = 0 if torch.cuda.is_available() else "cpu"
batch = 16 if torch.cuda.is_available() else 8

def evaluar(ruta, nombre):
    print(f"\nEvaluando: {nombre}")

    modelo = YOLO(str(ruta))

    metricas = modelo.val(
        data=str(DATA_DIR),
        split="test",
        imgsz=256,
        batch=batch,
        device=device,
        project="/content/runs/comparacion_mariposas",
        name=nombre,
        exist_ok=True,
        plots=True
    )

    return metricas.top1, metricas.top5

# Evaluar ambos con las mismas 500 imágenes de prueba
original_top1, original_top5 = evaluar(
    MODELO_ORIGINAL, "modelo_original"
)

nuevo_top1, nuevo_top5 = evaluar(
    MODELO_NUEVO, "modelo_reentrenado"
)

# Comparar resultados
print("\n========== COMPARACIÓN FINAL ==========")
print(f"Original - Top-1:       {original_top1:.2%}")
print(f"Reentrenado - Top-1:    {nuevo_top1:.2%}")
print(f"Diferencia Top-1:       {(nuevo_top1 - original_top1) * 100:+.2f} puntos porcentuales")

print(f"\nOriginal - Top-5:       {original_top5:.2%}")
print(f"Reentrenado - Top-5:    {nuevo_top5:.2%}")
print(f"Diferencia Top-5:       {(nuevo_top5 - original_top5) * 100:+.2f} puntos porcentuales")

if nuevo_top1 > original_top1:
    print("\nRESULTADO: mejoró Top-1 en las imágenes de prueba.")
elif nuevo_top1 < original_top1:
    print("\nRESULTADO: el modelo original obtuvo mejor Top-1.")
else:
    print("\nRESULTADO: ambos obtuvieron el mismo Top-1.")


Evaluando: modelo_original
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-cls summary (fused): 47 layers, 1,538,834 parameters, 0 gradients, 3.2 GFLOPs
train: /content/gdrive/.shortcut-targets-by-id/1dBL1-8SLvA31GaEontBKs4CGj5TsFZxJ/data/train... found 3500 images in 10 classes ✅ 
val: /content/gdrive/.shortcut-targets-by-id/1dBL1-8SLvA31GaEontBKs4CGj5TsFZxJ/data/val... found 1000 images in 10 classes ✅ 
test: /content/gdrive/.shortcut-targets-by-id/1dBL1-8SLvA31GaEontBKs4CGj5TsFZxJ/data/test... found 500 images in 10 classes ✅ 
WARNING ⚠️ test: Slow image access detected (ping: 0.8±0.2 ms, read: 0.6±0.2 MB/s, size: 396.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
test: Scanning /content/gdrive/.shortcut-targets-by-id/1dBL1-8SLvA31GaEontBKs4CGj5TsFZxJ/data/test... 500 images, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 9.4Mit/s 0.0s
               cla

# **Matrices de comparación:**

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

BASE = Path('/content/runs/comparacion_mariposas')
ZIP_PATH = Path('/content/matrices_comparacion_test.zip')

carpetas = {
    'modelo_original': BASE / 'modelo_original',
    'modelo_reentrenado': BASE / 'modelo_reentrenado'
}

encontradas = []

for nombre, carpeta in carpetas.items():
    print(f'\n{nombre}:')

    if not carpeta.is_dir():
        print('No se encontró la carpeta.')
        continue

    for archivo in sorted(carpeta.glob('*confusion_matrix*.png')):
        print('-', archivo.name)
        encontradas.append((nombre, archivo))

if encontradas:
    with ZipFile(ZIP_PATH, 'w', ZIP_DEFLATED) as zipf:
        for nombre, archivo in encontradas:
            zipf.write(archivo, f'{nombre}/{archivo.name}')

    print(f'\nMatrices reunidas: {len(encontradas)}')
    files.download(str(ZIP_PATH))
else:
    print(
        '\nNo se encontraron matrices en las carpetas '
        'de comparación.'
    )


modelo_original:
- confusion_matrix.png
- confusion_matrix_normalized.png

modelo_reentrenado:
- confusion_matrix.png
- confusion_matrix_normalized.png

Matrices reunidas: 4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Entrenamiento BASE:**

In [ ]:
import os
import torch

from ultralytics import YOLO

model = YOLO(
    DATA_DIR + "/runs/mariposas/weights/best.pt"
)

# Ajustar el entrenamiento al hardware disponible.


device = 0 if gpu_disponible else "cpu"
epochs = 50 if gpu_disponible else 3
batch = 16 if gpu_disponible else 8

print("Dispositivo:", device)
print("Épocas:", epochs)
print("Carpeta del dataset:", DATA_DIR)

# Entrenar con las divisiones ya existentes.
results = model.train(
    data=str(DATA_DIR),
    epochs=epochs,
    imgsz=256,
    batch=batch,
    device=device,
    patience=3,
    project=DATA_DIR + "/runs",
    name="mariposas",
    exist_ok=True
)

Dispositivo: 0
Épocas: 10
Carpeta del dataset: /content/gdrive/MyDrive/data
Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gdrive/MyDrive/data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0.937, mosaic=1.0, mul

# **TESTEO:**

In [ ]:
from ultralytics import YOLO

# Cargar el mejor modelo guardado.
modelo_test = YOLO(
    "/content/runs/mariposas/weights/best.pt"
)

# Evaluar las imágenes reservadas para prueba.
metricas_test = modelo_test.val(
    data=str(DATA_DIR),
    split="test",
    imgsz=256,
    batch=16,
    device=0
)

print(
    f"Exactitud Top-1 en TEST: "
    f"{metricas_test.top1:.2%}"
)

print(
    f"Exactitud Top-5 en TEST: "
    f"{metricas_test.top5:.2%}"
)

Ultralytics 8.4.190 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-cls summary (fused): 47 layers, 1,538,834 parameters, 0 gradients, 3.2 GFLOPs
train: /content/gdrive/.shortcut-targets-by-id/1dBL1-8SLvA31GaEontBKs4CGj5TsFZxJ/data/train... found 3500 images in 10 classes ✅ 
val: /content/gdrive/.shortcut-targets-by-id/1dBL1-8SLvA31GaEontBKs4CGj5TsFZxJ/data/val... found 1000 images in 10 classes ✅ 
test: /content/gdrive/.shortcut-targets-by-id/1dBL1-8SLvA31GaEontBKs4CGj5TsFZxJ/data/test... found 500 images in 10 classes ✅ 
WARNING ⚠️ test: Slow image access detected (ping: 0.8±0.2 ms, read: 1.7±0.7 MB/s, size: 396.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
test: Scanning /content/gdrive/.shortcut-targets-by-id/1dBL1-8SLvA31GaEontBKs4CGj5TsFZxJ/data/test... 500 images, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 3.8it/s 2:11
test: New cache created: /content/gdrive/.short

# **Guardar en el Drive:**

In [ ]:
from pathlib import Path
import shutil

# Carpeta donde YOLO guardó el entrenamiento.
ORIGEN = Path("/content/runs/mariposas")

# Carpeta permanente en tu Google Drive.
DESTINO = Path("/content/gdrive/MyDrive/data")

DESTINO.mkdir(parents=True, exist_ok=True)

# Copiar los dos modelos.
for nombre in ["best.pt", "last.pt"]:
    archivo = ORIGEN / "weights" / nombre

    if archivo.exists():
        shutil.copy2(archivo, DESTINO / nombre)
        print(f"Guardado: {DESTINO / nombre}")
    else:
        print(f"No se encontró: {archivo}")

# Guardar también el registro del entrenamiento.
for nombre in ["results.csv", "results.png"]:
    archivo = ORIGEN / nombre

    if archivo.exists():
        shutil.copy2(archivo, DESTINO / nombre)
        print(f"Guardado: {DESTINO / nombre}")

print("\nProceso de guardado terminado.")

Guardado: /content/gdrive/MyDrive/data/best.pt
Guardado: /content/gdrive/MyDrive/data/last.pt
Guardado: /content/gdrive/MyDrive/data/results.csv
Guardado: /content/gdrive/MyDrive/data/results.png

Proceso de guardado terminado.


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/gdrive/MyDrive/Optativa1/modelo/data/best.pt")
# Exportar a ONNX (opcional, recomendado para producción)
model.export(format="onnx")

Ultralytics 8.4.193 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n-cls summary (fused): 47 layers, 1,538,834 parameters, 0 gradients, 0.5 GFLOPs

PyTorch: starting from '/content/gdrive/MyDrive/Optativa1/modelo/data/best.pt' with input shape (1, 3, 256, 256) BCHW and output shape(s) (1, 10) (3.1 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 217ms
Prepared 4 packages in 575ms
Installed 4 packages in 46ms
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.31.1
 + onnxslim==0.1.98

requirements: AutoUpdate success ✅ 1.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: sli

'/content/gdrive/MyDrive/Optativa1/modelo/data/best.onnx'